In [1]:
import numpy as np
from sentence_transformers import SentenceTransformer

# Module-level model (lazy loaded)
_model = None

def search_index(query: str, faiss_index, chunks: list[str], top_k: int = 3) -> list[str]:
    """
    Searches the FAISS index using a query and returns top_k matching chunks.

    Args:
        query (str): Search query.
        faiss_index: FAISS index object.
        chunks (list[str]): List of original text chunks.
        top_k (int): Number of results to return.

    Returns:
        list[str]: Top matching chunk strings.
    """
    global _model

    # Lazy load model
    if _model is None:
        _model = SentenceTransformer("all-MiniLM-L6-v2")

    # Generate query embedding
    query_embedding = _model.encode([query], show_progress_bar=False)

    # Convert to numpy float32 with shape (1, 384)
    query_vec = np.array(query_embedding).astype("float32").reshape(1, -1)

    # Search FAISS index
    distances, indices = faiss_index.search(query_vec, top_k)

    # Extract valid results (filter out -1)
    results = []
    for idx in indices[0]:
        if idx != -1 and 0 <= idx < len(chunks):
            results.append(chunks[idx])

    return results

c:\Users\SHREE\miniconda3\envs\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def build_prompt(context_chunks: list[str], question: str) -> str:
    """
    Builds a structured prompt with system instructions and user context.

    Args:
        context_chunks (list[str]): Retrieved text chunks.
        question (str): User's question.

    Returns:
        str: Complete prompt string.
    """

    # System instruction (max 5 sentences)
    system_instruction = (
        "You are a document assistant that answers questions ONLY using the provided context. "
        "Do not use outside knowledge or make assumptions. "
        "If the answer is not present in the context, say: \"I cannot find that information in the document.\" "
        "Always cite which context section (e.g., Context [1]) supports your answer. "
        "Keep your response clear and concise."
    )

    # Build context section
    context_text = ""
    for i, chunk in enumerate(context_chunks, start=1):
        context_text += f"Context [{i}]: {chunk}\n"

    # User section
    user_section = f"{context_text}\nQuestion: {question}"

    # Final prompt
    prompt = f"{system_instruction}\n\n{user_section}"

    return prompt

In [ ]:
import os
from openai import OpenAI, APIError

# similar available models
# "openai/gpt-oss-20b"
# "openai/gpt-oss-120b"
# "llama-3.1-8b-instant"
# "llama-3.3-70b-versatile"
# "qwen/qwen3-32b"

def query_llm(
    prompt: str,
    model: str = "openai/gpt-oss-20b",
    max_tokens: int = 512
) -> str:
    client = OpenAI(
        base_url="https://api.groq.com/openai/v1",
        api_key=os.environ.get("GROQ_API_KEY"),
    )

    try:
        response = client.chat.completions.create(
            model=model,
            max_tokens=max_tokens,
            messages=[
                {"role": "user", "content": prompt}
            ],
        )
        return response.choices[0].message.content
    except APIError as e:
        return "LLM error: " + str(e)
